# Compilation

To be fast, Julia needs to **specialize** code, that is **compile specific native versions of the code** utilizing the type information. **The better the specialization the faster the code!**

## "Just ahead of time" compilation

* Julia **specializes on the types of function arguments** and 
* compiles efficient machine code **when a function is called for the first time** (with these input argument types).

If the same function is called again with the same input argument types, the already existing machine code is reused.


In [1]:
func(x,y) = 2x + y

func (generic function with 1 method)

In [2]:
x = [1.2, 3.4, 5.6] # Vector{Float64}
y = [0.4, 0.7, 0.9] # Vector{Float64}

@time func(x,y);
@time func(x,y);

  0.152581 seconds (244.85 k allocations: 13.526 MiB, 99.98% compilation time)
  0.000005 seconds (4 allocations: 160 bytes)


**First call:** compilation + running the code

**Second call:** running the code


In [3]:
@time func(x,y);

  0.000007 seconds (4 allocations: 160 bytes)


If one of the input types changes, Julia compiles a new specialization of the function!


In [4]:
typeof(x)

Vector{Float64} (alias for Array{Float64, 1})

In [5]:
x = [1, 3, 5]

3-element Vector{Int64}:
 1
 3
 5

In [6]:
typeof(x)

Vector{Int64} (alias for Array{Int64, 1})

In [7]:
@time func(x,y); # Vector{Int64}, Vector{Float64}
@time func(x,y);

  0.149721 seconds (221.83 k allocations: 12.306 MiB, 99.98% compilation time)
  0.000005 seconds (4 allocations: 160 bytes)


We now have two efficient native codes in the cache: one for all `Vector{Float64}` inputs and another one for `Vector{Int64}` as the first and `Vector{Float64}` as the second argument type.

In [8]:
methods(func)

# 1 method for generic function "func" from Main:
 [1] func(x, y)
     @ In[1]:1

In [9]:
using MethodAnalysis
methodinstances(func)

2-element Vector{Core.MethodInstance}:
 MethodInstance for func(::Vector{Float64}, ::Vector{Float64})
 MethodInstance for func(::Vector{Int64}, ::Vector{Float64})

### Compilation pipeline

<p><br><img src="imgs/Julia_compilation_pipeline.svg" width="512"/></p>

* **AST**: abstract syntax tree
* **IR**: intermediate representation

More about Julia compilation, see [Bezanson J et al (2018) Julia: dynamism and performance reconciled by design. Proc ACM Program Lang.](https://doi.org/10.1145/3276490)

### What makes Julia fast?

**Specialization** → (Successful) **Type inference** → **Compilation**

## Introspection tools
#### (*But I really want to see what happens!*)

We can inspect the code at all transformation stages with a bunch of macros:

<img src="./imgs/julia_introspection_macros.svg" width=300px>

In [10]:
@macroexpand @show 3+3

quote
    Base.println("3 + 3 = ", Base.repr(begin
                #= show.jl:1282 =#
                local var"#49#value" = 3 + 3
            end))
    var"#49#value"
end

In [11]:
f(x, y) = x^3 + y/2

f (generic function with 1 method)

In [15]:
@code_lowered f(1.0,2.0)

CodeInfo(
1 ─ %1 = Main.:+
│   %2 = Main.:^
│   %3 =   builtin Core.apply_type(Base.Val, 3)
│   %4 =   dynamic (%3)()
│   %5 =   dynamic Base.literal_pow(%2, x, %4)
│   %6 = Main.:/
│   %7 =   dynamic (%6)(y, 2)
│   %8 =   dynamic (%1)(%5, %7)
└──      return %8
)

*NOTE*: technically we want nothing dynamic as above


In [13]:
@code_typed f(1.0,2.0)

CodeInfo(
1 ─ %1 = intrinsic Base.mul_float(x, x)::Float64
│   %2 = intrinsic Base.mul_float(%1, x)::Float64
│   %3 = intrinsic Base.div_float(y, 2.0)::Float64
│   %4 = intrinsic Base.add_float(%2, %3)::Float64
└──      return %4
) => Float64

From the types of the input arguments, Julia has figured out all the intermediate types. This crucial process is known as **type inference** and its success is the basis for a good specialization (i.e. performant native code as a result). Moreover, the generic power function computing the cubic of `x` is replaced by specific floating-point multiplications (**static dispatch**).

In [17]:
@code_llvm debuginfo=:none f(1.0,2.0)

; Function Signature: f(Float64, Float64)
define double @julia_f_3087(double %"x::Float64", double %"y::Float64") local_unnamed_addr #0 {
top:
  %0 = fmul double %"x::Float64", %"x::Float64"
  %1 = fmul double %"x::Float64", %0
  %2 = fmul double %"y::Float64", 5.000000e-01
  %3 = fadd double %1, %2
  ret double %3
}


The expensive divide operation (`y/2`) is replaced by multiplying by 0.5. In the end, giving two `Float64` arguments this function has 4 floating-point operations, i.e. 3 multiplications and 1 addition, instead of cubic function and division.

In [ ]:
@code_native debuginfo=:none f(1.0,2.0)

Let's compare this to integer inputs.

In [ ]:
@code_native debuginfo=:none f(1,2)

## How important is specialization?

Let's try to estimate the performance gain by specialization. To prevent specialization, we deliberately throw away any useful type information and operate on a `Vector{Any}` that can literally store anything!

(This is qualitatively comparable to what Python does.)


In [19]:
func(v) = 2*v[1] + v[2] # version of func that takes in a vector
methods(func)


# 2 methods for generic function "func" from Main:
 [1] func(v)
     @ In[19]:1
 [2] func(x, y)
     @ In[1]:1

In [20]:
rand(2)

2-element Vector{Float64}:
 0.351010970358819
 0.5254909010585385

In [21]:
Any[rand(), rand()]

2-element Vector{Any}:
 0.3366535268710842
 0.8896072144655577

In [22]:
using BenchmarkTools

v_typed = rand(2)
v_any = Any[rand(), rand()]

@btime func($v_typed);
@btime func($v_any);

  3.061 ns (0 allocations: 0 bytes)
  43.250 ns (2 allocations: 32 bytes)


For benchmarking we generally use `@btime` (or `@benchmark`) from [BenchmarkTools.jl](https://github.com/JuliaCI/BenchmarkTools.jl). This will take care of a couple of things for us:
* Exclude first run.
* Run the code multiple times (→ statistics).
* Benchmark in a function (local scope).

**General rule:** For proper benchmarking don't use `@time` but `@btime` and interpolate (`$`) global input arguments.

(Prefixing variable with `$` always means interpolation in Julia, e.g. string interpolation.)

In [23]:
hlrs = "HLRS"
welcome = "Welcome to $hlrs"

"Welcome to HLRS"

In [24]:
@benchmark func($v_any)

BenchmarkTools.Trial: 10000 samples with 982 evaluations per sample.
 Range (min … max):  43.893 ns … 637.680 ns  ┊ GC (min … max): 0.00% … 85.32%
 Time  (median):     49.608 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   54.707 ns ±  25.638 ns  ┊ GC (mean ± σ):  2.70% ±  5.55%

   ▂▅▇█▇▆▅▂▃▃▂▂▂▂▃▄▄▂▁                ▁                        ▂
  █████████████████████████▇▇▇▇▆▆▅▆▆▇▇█▇▇█▇▇▅▄▅▅▄▄▅▆▄▆▆▆▅▄▃▅▅▄ █
  43.9 ns       Histogram: log(frequency) by time       109 ns <

 Memory estimate: 32 bytes, allocs estimate: 2.

In [ ]:
@code_typed func(rand(2))

**static dispatch**: the generic functions `*` and `+` are replaced by specific implementations.

In [ ]:
@code_typed func(Any[rand(), rand()])

Note here the generic functions `*` and `+` can not be replaced by specific variants due to lack of type information. This leads to inefficient **runtime dispatch**.

## Dispatch and specialization

**Types drive both dispatch and specialization.**

First, the most specific method is selected (dispatch), then it gets compiled to efficient native code (specialization). Let's reconsider our earlier example:

In [ ]:
myabs(x::Real) = sign(x) * x
myabs(z::Complex) = sqrt(real(z * conj(z)))

In [ ]:
@code_native myabs(3.2 + 4.5im) # complex input

In [ ]:
@code_native myabs(3 + 4im) # also complex input but different native code (due to specialization)!

## Precompilation

Besides the "just ahead of time" compilation discussed above, **Julia precompiles packages and stores the resulting binary code (among other things).**

In [25]:
Base.compilecache_dir(Base.PkgId(BenchmarkTools))

"/home/hlrskn16/.julia/compiled/v1.13/BenchmarkTools"

In [ ]:
readdir(Base.compilecache_dir(Base.PkgId(BenchmarkTools)))

### A note for heterogeneous HPC clusters

By default, precompilation produces native code for the CPU type it is running on. This means that it uses the [Instruction Set Architecture (ISA)](https://en.wikipedia.org/wiki/Instruction_set_architecture) of this CPU.

This can lead to issues on heterogeneous clusters where different nodes have different CPU types. E.g. you precompile Julia packages on a login node with an Intel CPU but want to run the code on a compute node with AMD CPUs.

**Solution: Multiversioning**

```julia
# HLRS Training Cluster
export JULIA_CPU_TARGET="generic;cascadelake,clone_all;skylake-avx512,clone_all;icelake-server,clone_all"
```

This will compile a generic (but slow) version as well as efficient variants for Intel Sandybridge and Intel Cascade Lake and Intel Skylake CPUs.

# Core messages of this Notebook

* **A function is compiled when called for the first time** with a given set of argument types.
* There are **multiple code transformation steps** which can be inspected through macros like `@code_warntype` or `@descend` from Cthulhu.jl.
* What makes Julia fast? Successful **Type inference** → **Specialization** → **Compilation**.
* Functions should almost always be benchmarked with **BenchmarkTools.jl's `@btime` and `@benchmark`** instead of `@time`.
* In virtually all cases, **explicit type annotations are irrelevant for performance**.